# COMP7103 Lab 3 · Clean a messy customer table, twice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab3/Lab3_cleaning.ipynb)

100 rows of a messy customer table: typos, mixed formats, mixed units, missing values and customers who registered twice. You clean it **by hand with pandas**, then clean **one text column with an LLM**, and score both against a hidden clean version.

| Time | What you do |
|---|---|
| 5 min | Brief: look at the mess and the cleaning rules |
| 30 min | Clean by hand, then one column with an LLM, and score both |
| 10 min | Share-out |

This lab is ungraded practice for the exams. All people and companies in the table are made up.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the messy table, the cleaning rules and the LLM prompts.

In [ ]:
import os, re, json, time, urllib.request
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)

# A blank (___) that you have not filled in stops with: name '___' is not defined
# (after a dot, as in .str.___(), it stops with: has no attribute '___')
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab3/data/"

def get(name):
    """Use the local file if it is there (Lab 3 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 3 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab3")
    return os.path.join("data", name)

messy = pd.read_csv(get("messy_customers.csv"), dtype=str, keep_default_na=False)
rules = pd.read_csv(get("cleaning_rules.csv"))
prompts = json.load(open(get("llm_prompts.json")))
print(messy.shape[0], "rows ×", messy.shape[1], "columns")

The scoring code (given). It reads `clean_customers.csv`, the hidden clean table. Don't open that file: let `score()` use it.

In [ ]:
def score(df):
    """Compare a cleaned table with the hidden clean table. Returns the error rate per column (%).
    A row that should be there but is missing counts as wrong in every column."""
    truth = pd.read_csv(get("clean_customers.csv"), dtype=str, keep_default_na=False).set_index("customer_id")
    d = df.copy()
    d["customer_id"] = d["customer_id"].astype(str)
    d = d.set_index("customer_id")
    extra, missing = set(d.index) - set(truth.index), set(truth.index) - set(d.index)
    print(f"Rows kept: {len(d)} · duplicates still in: {len(extra)} · rows dropped by mistake: {len(missing)}")
    out = {}
    for col in truth.columns:
        if col not in d.columns:
            out[col] = np.nan
            continue
        a = d[col].reindex(truth.index)
        if col == "parcel_weight_kg":
            ok = (pd.to_numeric(a, errors="coerce") - truth[col].astype(float)).abs() <= 0.02
        else:
            ok = a.fillna("").astype(str).str.strip() == truth[col]
        out[col] = round(100 * (1 - ok.mean()), 1)
    return pd.Series(out, name="error rate (%)")

## 1 · Look at the mess (5 min)

In [ ]:
messy.head(10)

The cleaning rules: the target format for each column.

In [ ]:
rules

The company column before cleaning: one company, many spellings.

In [ ]:
messy["company"].value_counts()

## 2 · Clean by hand with pandas (about 20 min)

Run the timer cell first. Then work through Checks A to F. Checks D and E also time themselves, from their first run to their last: the share-out compares these minutes with the LLM's.

In [ ]:
t_start = globals().get("t_start", time.time())   # timer for the whole by-hand cleaning (starts on the first run only)
clean = messy.copy()

**Check A · names, phones and emails.**

In [ ]:
clean["name"] = clean["name"].str.split().str.join(" ").str.___()           # single spaces, Title Case
clean["phone"] = clean["phone"].str.replace(r"\D", "", regex=True).str[-___:]  # keep the last 8 digits
clean["email"] = clean["email"].str.strip().str.lower().replace({"n/a": "", "-": "", "unknown": ""})
clean[["name", "phone", "email"]].head()

**Check B · dates.** Four formats appear: 2024-03-05, 05/03/2024, 05 Mar 2024 and 05.03.2024. All are day first. Fill in the last format.

In [ ]:
def to_iso(s):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y", ___):
        try:
            return pd.to_datetime(s.strip(), format=fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return ""

clean["signup_date"] = clean["signup_date"].map(to_iso)
clean["signup_date"].head()

**Check C · weights.** Convert everything to kg: 1 lb = 0.45359237 kg, 1,000 g = 1 kg.

In [ ]:
def to_kg(s):
    s = s.replace(",", "").strip().lower()
    number = float(re.findall(r"[\d.]+", s)[0])
    if s.endswith("kg"):
        return round(number, 2)
    if s.endswith("lb"):
        return round(number * ___, 2)
    if s.endswith("g"):
        return round(number / ___, 2)

clean["parcel_weight_kg"] = messy["parcel_weight"].map(to_kg)
clean = clean.drop(columns="parcel_weight", errors="ignore")
clean["parcel_weight_kg"].describe()

**Check D · districts.** A rule fixes case and spacing. For the rest, add aliases (keys in lower case) until the last line prints an empty list.

In [ ]:
OFFICIAL = ['Central and Western', 'Wan Chai', 'Eastern', 'Southern', 'Yau Tsim Mong', 'Sham Shui Po', 'Kowloon City', 'Wong Tai Sin', 'Kwun Tong', 'Kwai Tsing', 'Tsuen Wan', 'Tuen Mun', 'Yuen Long', 'North', 'Tai Po', 'Sha Tin', 'Sai Kung', 'Islands']
t_D = globals().get("t_D", time.time())   # timer for Check D (starts on the first run only)

ALIASES = {"ytm": "Yau Tsim Mong", "kln city": "Kowloon City", "wts": ___}   # add the ones you find

def fix_district(s):
    s = s.strip()
    if s.lower() in ("", "n/a", "-", "unknown"):
        return ""
    for d in OFFICIAL:                                   # the same name in another case or spacing
        if s.lower().replace(" ", "") == d.lower().replace(" ", ""):
            return d
    return ALIASES.get(s.lower(), s)

clean["district"] = clean["district"].map(fix_district)
minutes_D = (time.time() - t_D) / 60
sorted(set(clean["district"]) - set(OFFICIAL) - {""})   # still not official?

**Check E · companies.** A rule fixes case, spacing and the suffix. Then look at what is left and fix the typos.

In [ ]:
t_E = globals().get("t_E", time.time())   # timer for Check E (starts on the first run only)

def fix_company(s):
    s = " ".join(s.split()).title()                          # spacing and case
    s = re.sub(r"\s+(Co\.\s+)?(Ltd\.?|Limited)$", "", s)  # drop the suffix ...
    return s + " Ltd"                                        # ... and add the standard one

clean["company"] = clean["company"].map(fix_company)
clean["company"].value_counts()

In [ ]:
TYPOS = {"Victoria Pirnt Works Ltd": "Victoria Print Works Ltd"}   # add the others you find, as "wrong": "right"
clean["company"] = clean["company"].replace(TYPOS)
minutes_E = (time.time() - t_E) / 60
clean["company"].value_counts()

**Check F · customers who registered twice.** The same person has the same name and phone. Keep the first registration.

In [ ]:
clean = clean.sort_values("customer_id")
clean = clean.drop_duplicates(subset=[___, ___], keep="first")
minutes_by_hand = (time.time() - t_start) / 60      # stop the timer
print(len(clean), "rows ·", round(minutes_by_hand, 1), "minutes")

**Score your by-hand cleaning.**

In [ ]:
by_hand = score(clean)
by_hand

## 3 · Clean one column with an LLM (about 10 min)

1. Choose the column: `district` or `company`.
2. Run the next cell. It prints a message: the instructions and the column's distinct values, numbered. It also starts a timer.
3. Paste the message into any chat LLM you use. The data are made up, so it is fine to share them.
4. Copy the LLM's whole answer with the chat's Copy button (selecting the text with the mouse can lose the numbers). Paste it between the triple quotes in the cell after, and run it.

No LLM available? Use the ready-made answer instead (see the comment in that cell). It comes from Qwen2.5-1.5B-Instruct, a small open model, asked one value at a time. Its mistakes, stray formatting included, are real model output. Bigger chat models usually do better: measure yours.

In [ ]:
COLUMN = "district"        # or "company"

def llm_values(column):
    return sorted(messy[column].fillna("").str.strip().unique())

def llm_message(column):
    lines = "\n".join(f"{i}. {v or '(empty)'}" for i, v in enumerate(llm_values(column), 1))
    return prompts[column] + "\n\n" + lines

print(llm_message(COLUMN))
t_llm = time.time()        # timer for the LLM cleaning

In [ ]:
llm_reply = """
paste the LLM's answer here
"""
# No LLM? Use the ready-made answer instead:
# llm_reply = open(get(f"llm_fallback_{COLUMN}.txt")).read()
minutes_llm = (time.time() - t_llm) / 60

Read the answer, put it into the table, and score the column (given). Only the chosen column changes; the rest is your by-hand table.

In [ ]:
def read_answers(reply):
    """Numbered lines "N. name" -> {N: name}. Tolerates bold or code formatting, quotes and "old → new"."""
    answers = {}
    for line in reply.strip().splitlines():
        m = re.match(r"\s*[*_`]*(\d+)[.)][*_`]*\s*(.*)$", line)
        if m:
            a = m.group(2).strip().strip('"*`_ ')
            a = re.split(r"\s*(?:→|->)\s*", a)[-1].strip('"*`_ ')
            answers[int(m.group(1))] = "" if a.lower() in ("(empty)", "empty") else a
    return answers

def apply_llm(column, reply):
    answers = read_answers(reply)
    if not answers:
        raise SystemExit("No numbered lines found: paste the LLM's whole answer, numbers included.")
    mapping = {v: answers.get(i, v) for i, v in enumerate(llm_values(column), 1)}
    print(f"{len(answers)} answers read for {len(mapping)} values")
    if len(answers) != len(mapping):
        print("Warning: expected", len(mapping), "numbered lines. Check the paste: a skipped or renumbered line shifts the answers.")
    out = clean.copy()
    raw = messy.set_index("customer_id")[column].str.strip()
    out[column] = out["customer_id"].map(raw).map(mapping)
    return out

with_llm = apply_llm(COLUMN, llm_reply)
compare = pd.DataFrame({"by hand": by_hand, "LLM column": score(with_llm)})
compare.loc[[COLUMN]]

## 4 · Results and share-out (10 min)

In [ ]:
pd.DataFrame({
    "column": [COLUMN, COLUMN],
    "method": ["by hand (pandas)", "LLM"],
    "error rate (%)": [compare.loc[COLUMN, "by hand"], compare.loc[COLUMN, "LLM column"]],
    "minutes": [round({"district": minutes_D, "company": minutes_E}[COLUMN], 1), round(minutes_llm, 1)],
})

By hand, the minutes are those of Check D (district) or Check E (company), so both methods are timed on the same column.

- Report your error rate and minutes for the column you cleaned both ways; we collect them on the class board.
- Where did the LLM help? What did it get wrong, and would you have noticed without the score?
- The rule from the lecture: every LLM-cleaned column gets a measured error rate on a sample checked by hand.